In [5]:
!pip install xgboost

In [6]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
from sklearn.svm import SVC
import pandas as pd
from data_preprocessing_module import * #our own module


# Load dataset
df = pd.read_csv('datasets/changed/data_news_1.csv')
df['label'] = df['type_of_news'].map({'Fake': 0, 'Real': 1})

df['title_clean'] = df['title'].apply(preprocess_text)

# Use preprocessed title
X = df['title_clean']
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tfidf vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_df=0.7)

# Hyperparameter tuning for Random Forest
rf_pipeline = make_pipeline(tfidf, RandomForestClassifier(random_state=42))
rf_params = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [None, 10, 20]
}
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=3, scoring='f1', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

# Define models
models = {
    'Naive Bayes': make_pipeline(tfidf, MultinomialNB()),
    'Logistic Regression': make_pipeline(tfidf, LogisticRegression(max_iter=300)),
    'Random Forest (Tuned)': best_rf,
    'Gradient Boosting': make_pipeline(tfidf, GradientBoostingClassifier(random_state=42)),
    'XGBoost': make_pipeline(tfidf, XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
    'SVM': make_pipeline(tfidf, SVC(probability=True, random_state=42))  # Add SVM
}

# Evaluation
model_average_scores = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}:")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    model_average_scores[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Soft Voting Ensemble
ensemble_model = VotingClassifier(estimators=[
    ('nb', models['Naive Bayes']),
    ('lr', models['Logistic Regression']),
    ('rf', models['Random Forest (Tuned)']),
    ('gb', models['Gradient Boosting']),
    ('xgb', models['XGBoost']),
    ('svm', models['SVM'])  # Add SVM to the ensemble
], voting='soft')

ensemble_model.fit(X_train, y_train)
ensemble_pred = ensemble_model.predict(X_test)

ensemble_accuracy = accuracy_score(y_test, ensemble_pred)
ensemble_precision = precision_score(y_test, ensemble_pred)
ensemble_recall = recall_score(y_test, ensemble_pred)
ensemble_f1 = f1_score(y_test, ensemble_pred)

print("\nVoting Ensemble Model Performance:")
print(f"Accuracy: {ensemble_accuracy:.4f}")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall: {ensemble_recall:.4f}")
print(f"F1 Score: {ensemble_f1:.4f}")

# Stacking Classifier
stacking_model = StackingClassifier(
    estimators=[
        ('nb', models['Naive Bayes']),
        ('lr', models['Logistic Regression']),
        ('rf', models['Random Forest (Tuned)']),
        ('gb', models['Gradient Boosting']),
        ('svm', models['SVM'])  # Add SVM to stacking
    ],
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

stacking_model.fit(X_train, y_train)
stack_pred = stacking_model.predict(X_test)

stack_accuracy = accuracy_score(y_test, stack_pred)
stack_precision = precision_score(y_test, stack_pred)
stack_recall = recall_score(y_test, stack_pred)
stack_f1 = f1_score(y_test, stack_pred)

print("\nStacking Model Performance:")
print(f"Accuracy: {stack_accuracy:.4f}")
print(f"Precision: {stack_precision:.4f}")
print(f"Recall: {stack_recall:.4f}")
print(f"F1 Score: {stack_f1:.4f}")

# Compare all models
print("\nModel Accuracy Comparison:")
for model_name, scores in model_average_scores.items():
    print(f"{model_name}: Accuracy: {scores['Accuracy']:.4f}")
print(f"Voting Ensemble: Accuracy: {ensemble_accuracy:.4f}")
print(f"Stacking Classifier: Accuracy: {stack_accuracy:.4f}")


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kushalpanthi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kushalpanthi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Evaluating Naive Bayes:
Accuracy: 0.7297
Precision: 0.8000
Recall: 0.6316
F1 Score: 0.7059

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.83      0.75        18
           1       0.80      0.63      0.71        19

    accuracy                           0.73        37
   macro avg       0.74      0.73      0.73        37
weighted avg       0.74      0.73      0.73        37


Evaluating Logistic Regression:
Accuracy: 0.6216
Precision: 0.6471
Recall: 0.5789
F1 Score: 0.6111

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.67      0.63        18
           1       0.65      0.58      0.61        19

    accuracy                           0.62        37
   macro avg       0.62      0.62      0.62        37
weighted avg       0.62      0.62      0.62        37


Evaluating Random Forest (Tuned):
Accuracy: 0.6216
Precision: 0.6087
Recall: 0.7368
F1 Score: 0.6667

Cla

/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [17:33:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.6486
Precision: 0.6000
Recall: 0.9474
F1 Score: 0.7347

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.33      0.48        18
           1       0.60      0.95      0.73        19

    accuracy                           0.65        37
   macro avg       0.73      0.64      0.61        37
weighted avg       0.73      0.65      0.61        37


Evaluating SVM:
Accuracy: 0.6486
Precision: 0.6667
Recall: 0.6316
F1 Score: 0.6486

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.67      0.65        18
           1       0.67      0.63      0.65        19

    accuracy                           0.65        37
   macro avg       0.65      0.65      0.65        37
weighted avg       0.65      0.65      0.65        37



/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [17:33:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Voting Ensemble Model Performance:
Accuracy: 0.5946
Precision: 0.5909
Recall: 0.6842
F1 Score: 0.6341

Stacking Model Performance:
Accuracy: 0.6216
Precision: 0.6316
Recall: 0.6316
F1 Score: 0.6316

Model Accuracy Comparison:
Naive Bayes: Accuracy: 0.7297
Logistic Regression: Accuracy: 0.6216
Random Forest (Tuned): Accuracy: 0.6216
Gradient Boosting: Accuracy: 0.6757
XGBoost: Accuracy: 0.6486
SVM: Accuracy: 0.6486
Voting Ensemble: Accuracy: 0.5946
Stacking Classifier: Accuracy: 0.6216


Naive Bayes
Accuracy: 72.97% – Best among all.

Precision (Real): 0.80 – When it predicts “Real”, it's right 80% of the time.

Recall (Real): 0.63 – It catches 63% of all real news in the test set.

F1 Score: 0.71 – Balanced performance.

Interpretation: It favours precision for Real news but misses some (lower recall). It performs well despite its simplicity, likely because Naive Bayes handles text (via TF-IDF) naturally well.


#naive bayes game the best score while taking the paramters as title becaue of the following:
You're using only the title (very short text), so:

The number of features is not enormous.

Logistic Regression and trees might overfit or underfit due to the short length.

Naive Bayes can catch general trends in word usage that separate fake vs real news quickly.

======
Accuracy: Measures the overall correctness of the model, i.e., the proportion of correct predictions.

Precision: Indicates how many of the predicted real news articles were actually real.

Recall: Shows how many of the actual real news articles were correctly identified by the model.

F1 Score: Balances precision and recall, providing a single metric that considers both false positives and false negatives.
